In [1]:
import re
import json
import html
import datetime 
import requests
import pandas as pd 
import urllib.request
from bs4 import BeautifulSoup


HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
             'cookie': 'over18=1;'}


In [2]:
# 'https://www.yourator.co/companies/mexcglobal/jobs/26192'

def get_job_detail(url_jobID):#='https://www.yourator.co/companies/PicCollage/jobs/10227'): 

    r = requests.get(url_jobID, headers=HEADERS)
    r.encoding = 'utf-8' #轉換編碼至UTF-8
    soup = BeautifulSoup(r.text)
    
    # 工作內容 條件要求 
    judge = soup.select("div[class='col-md-9 col-sm-12 job__content']")
    
#     job_content=[]
#     job_requirement=[]
    for i in range(len(judge)):
        Text = judge[i].text
        
        if '遠端型態' in Text:
            targetText = Text.split('遠端型態')[0]
            content_requirement_split = targetText.split('條件要求')
            
        elif '加分條件' in Text:
            targetText = Text.split('加分條件')[0]
            content_requirement_split = targetText.split('條件要求')
            
        elif '員工福利' in Text:
            targetText = Text.split('員工福利')[0]
            content_requirement_split = targetText.split('條件要求')
        
        elif '薪資範圍' in Text:
            targetText = Text.split('薪資範圍')[0]
            content_requirement_split = targetText.split('條件要求')
            
        content_requirement_split[0] = content_requirement_split[0].replace('\n','')
        content_requirement_split[0] = content_requirement_split[0].replace('\xa0','')
        content_requirement_split[0] = content_requirement_split[0].replace('工作內容','')
        jobContent = content_requirement_split[0]
        if content_requirement_split[1]!='\n\n':
            content_requirement_split[1] = content_requirement_split[1].replace('\n','')
            content_requirement_split[1] = content_requirement_split[1].replace('\xa0','')
            jobRequirement = content_requirement_split[1]
        else:
            jobRequirement = '-'
        
    return jobContent, jobRequirement
        
        
#get_job_detail()

In [3]:
# 欄位： 職缺內容, 公司名稱, 地址, 薪資, 網址, 應徵人數
# https://www.yourator.co/api/v2/job_categories
# https://www.yourator.co/api/v3/job_areas
# https://www.yourator.co/api/v2/jobs?position[]=full_time&sort=recent_updated&page=3

# 全職 最近更新時間
# URL = 'https://www.yourator.co/jobs?position[]=full_time&sort=newest'

# url_page = 'https://www.yourator.co/api/v2/jobs?position[]=full_time&sort=recent_updated&page=2'

def webCrawl_Yourator(pages):

    # columns
    job_title = []
    company_name = []
    address = []
    money = []
    job_content = []
    job_requirement = []
    trade = []
    tags = []
    job_detail_url = []

    url_page = f'https://www.yourator.co/api/v4/jobs?category[]=%E8%B3%87%E6%96%99%E5%B7%A5%E7%A8%8B%20%2F%20%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92&page=1&sort=recent_updated'
    #url_page = f'https://www.yourator.co/api/v2/jobs?position[]=full_time&sort=recent_updated'
    r = urllib.request.Request(url_page, headers=HEADERS)
    r = urllib.request.urlopen(r)

    soup = BeautifulSoup(r, 'html.parser')
    data = json.loads(str(soup))
    
    print(data['payload']['nextPage'], type(data['payload']['nextPage']))

    for j in data['payload']['jobs']:
        job_title.append(j['name'])
        company_name.append(j['company']['brand'])
        address.append(j['location'])
        money.append(j['salary'])
        #trade.append(j['category']['name'])
        temp = j['tags']

        if temp!=[]:
            temp = ', '.join(temp)
        else:
            temp='-'

        tags.append(temp)

        jobDetail_url = 'https://www.yourator.co'+j['path']
        
        Con, Req =  get_job_detail(jobDetail_url)
        job_content.append(Con)
        job_requirement.append(Req) 

        
    for p in range(2,pages):
        
        url_page = f'https://www.yourator.co/api/v4/jobs?category[]=%E8%B3%87%E6%96%99%E5%B7%A5%E7%A8%8B%20%2F%20%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92&page=1&sort=recent_updated&page={p}'
        #url_page = f'https://www.yourator.co/api/v2/jobs?position[]=full_time&sort=recent_updated&page={p}'
        r = urllib.request.Request(url_page, headers=HEADERS) #(url_page, headers=HEADERS)
        r = urllib.request.urlopen(r)

        soup = BeautifulSoup(r, 'html.parser')
        data = json.loads(str(soup))
        
        if data['payload']['nextPage'] != None: 
        
            print(data['payload']['nextPage'], type(data['payload']['nextPage']))

            for j in data['payload']['jobs']:
                job_title.append(j['name'])
                company_name.append(j['company']['brand'])
                address.append(j['location'])
                money.append(j['salary'])
                #trade.append(j['category']['name'])
                temp = j['tags']

                if temp!=[]:
                    temp = ', '.join(temp)
                else:
                    temp='-'

                tags.append(temp)

                jobDetail_url = 'https://www.yourator.co'+j['path']

                Con, Req =  get_job_detail(jobDetail_url)
                job_content.append(Con)
                job_requirement.append(Req) 
                
            #print('run')
                
        else:
            #print('stop')
            break

    final_df = pd.DataFrame()

    final_df['職缺名稱']=job_title
    final_df['公司名稱']=company_name
    final_df['工作內容']=job_content
    final_df['條件要求']=job_requirement
    final_df['工作地']=address
    final_df['薪資']=money
    #final_df['產業']=trade
    final_df['標籤']=tags

    return final_df


In [4]:
test = webCrawl_Yourator(7)
test

2 <class 'int'>
3 <class 'int'>
4 <class 'int'>


,職缺名稱,公司名稱,工作內容,條件要求,工作地,薪資,標籤
0,資料科學家,友達耘康股份有限公司,中醫只有華人才能精確掌握，雖然目前還很低調，但我們能提供的產品價值是過往廠商都無法提供的。除...,1.統計系畢業或具相關經驗2.具閱讀國外期刊及論文之能力3.熟悉模型開發流程（MLOps）4...,新竹市,面議（經常性薪資達4萬元）,"AI, Data Scientist, Statistics Analysis"
1,[總部] 機器學習主任工程師 Principal Engineer_0220*,Gamania 橘子集團,我們是集團的策略長室團隊，目前正在積極尋找Machine Learning 主任工程師 加入...,▍你需具有1.有 AI/ML 研發能力2.有 AI/ML 專案經驗3.優秀的溝通與團隊合作能...,臺北市,"NT$ 1,000,000 - 1,800,000 (年薪)","軟體開發, AI, 軟體設計工程師, 機器學習, 大數據, 演算法開發工程師"
2,Backend Engineer (Python),Dcard,【Please Apply Here】https://grnh.se/3f28ec0f1us...,We are looking for an excellent Backend Engine...,臺北市,面議（經常性薪資達4萬元）,-
3,AI 產品經理( Prompt Engineering/ Fine Tuning）,肖準行銷,部門介紹The Pocket CompanyThe Pocket Company是Accuc...,本科以上學位，資訊管理、資訊工程或相關專業優先考慮。擁有AI 產品經理或相關職位工作經驗，有...,臺北市,"NT$ 38,000 - 80,000 (月薪)",-
4,Data Engineer - Python,CTW株式会社,作為數據工程師（Python），您將負責在我們的G123平台上設計和實施強大的數據流程，以確...,- 計算機科學或其他相關領域的學士學位或以上學歷- 3年以上相關數據工程等經驗- 熟練掌握：...,臺北市,"NT$ 1,000,000 - (年薪)",Data Engineer
5,NLP Research Scientist,CTW株式会社,作為自然語言處理（NLP）研究技術人員，您將負責在與工程團隊密切合作的同時實施自然語言處理技...,- 在機器翻譯、自然語言生成、文本摘要、對話系統、問答等領域具有研究/行業經驗- 出色的分析...,臺北市,"NT$ 1,000,000 - (年薪)",NLP
6,MLOps Engineer,CTW株式会社,作為機器學習運維（MLOps）工程師，您將負責實現模型在生產系統中的自動化部署，以在G123...,在Python方面有可驗證的熟練技能熟練使用AWS及其服務，如SageMaker、EC2、E...,臺北市,"NT$ 1,000,000 - (年薪)",MachineLearning
7,Data Scientist,CTW株式会社,作為數據科學家，您將負責基於我們的G123平台大數據，應用機器學習、統計分析和數據可視化技術...,- 在統計分析、機器學習、自然語言處理（NLP）、推薦系統、自動化等相關領域具有經驗- 出色...,臺北市,"NT$ 1,000,000 - (年薪)",data s
8,Prompt Engineer,CTW株式会社,- 與人工智慧和數據科學團隊緊密合作，為AI語言模型（如GPT-4）或生成式AI模型（如St...,- 計算機科學、數據科學、語言學或相關領域的學士及以上學位- 有使用AI語言模型（如GPT-...,臺北市,"NT$ 1,000,000 - (年薪)",Prompt
9,CV Research Scientist,CTW株式会社,作為計算機視覺研究技術人員，您將負責在我們的內部橫幅生成工具中實現諸如GAN或穩定擴散等新的...,- 在圖像生成（GAN/穩定擴散）、神經圖像評估、物體檢測、圖像識別或類似領域具有研究/行業...,臺北市,"NT$ 1,000,000 - (年薪)",ComputerVision


In [5]:
# 中英分離

demand = test.filter(['職缺名稱','標籤','工作內容'])

# 创建一个新的DataFrame来存放拆分后的行
new_rows = []

# 遍历原始DataFrame
for index, row in demand.iterrows():
    job_titles = row['職缺名稱'].split('、')
    jobName = row['標籤']
    JD = row['工作內容']
    for title in job_titles:
        new_row = {'職稱': title, '標籤': jobName, '職缺描述':JD}
        new_rows.append(new_row)

# 创建包含新行的新DataFrame
new_demand = pd.DataFrame(new_rows)

# data_duty = ['數據分析師','資料科學家','資料工程師','AI工程師','演算法工程師']
# mask = new_demand['職務'].isin(data_duty)
# new_demand = new_demand[mask]

# 將 工作敘述 的網址取代掉
def remove_url_and_html(text):
    # 用 re 取代 URL
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    
    # 用 re 移除 HTML tag
    text = re.sub(r'<.*?>', '', text)
    text = text.replace('\n', '').replace('\r', '').replace('\t', '')
    
    return text

# use re to replace html entities
def replace_html_entities(text):
    
    text = re.sub(r'<.*?>', '', text)
    
    # parse entities
    text = html.unescape(text)
    
    return text

new_demand['職缺描述'] = new_demand['職缺描述'].apply(remove_url_and_html)
new_demand['職缺描述'] = new_demand['職缺描述'].apply(replace_html_entities)

# 中英文 分離
# 提取中文
def extract_chinese(text):
    chinese_pattern = r'[\u4e00-\u9fa5]+'  # 中文字符
    chinese_matches = re.findall(chinese_pattern, text)
    return ' '.join(chinese_matches)  # 將多個中文字符，以空格分隔

# 提取英文
def extract_english(text):
    english_pattern = r'[a-zA-Z]+'  # 英文字符
    english_matches = re.findall(english_pattern, text)
    return ' '.join(english_matches)  # 將多個英文字符，以空格分隔

new_demand['職缺描述（中文）'] = new_demand['職缺描述'].apply(extract_chinese)
new_demand['職缺描述（英文）'] = new_demand['職缺描述'].apply(extract_english)

new_demand


,職稱,標籤,職缺描述,職缺描述（中文）,職缺描述（英文）
0,資料科學家,"AI, Data Scientist, Statistics Analysis",中醫只有華人才能精確掌握，雖然目前還很低調，但我們能提供的產品價值是過往廠商都無法提供的。除...,中醫只有華人才能精確掌握 雖然目前還很低調 但我們能提供的產品價值是過往廠商都無法提供的 除...,Insight automated data pipelines
1,[總部] 機器學習主任工程師 Principal Engineer_0220*,"軟體開發, AI, 軟體設計工程師, 機器學習, 大數據, 演算法開發工程師",我們是集團的策略長室團隊，目前正在積極尋找Machine Learning 主任工程師 加入...,我們是集團的策略長室團隊 目前正在積極尋找 主任工程師 加入我們 你將負責 應用架構設計與實...,Machine Learning Gamania Data Center is an exp...
2,Backend Engineer (Python),-,【Please Apply Here】 are on a mission to spark ...,,Please Apply Here are on a mission to spark co...
3,AI 產品經理( Prompt Engineering/ Fine Tuning）,-,部門介紹The Pocket CompanyThe Pocket Company是Accuc...,部門介紹 是 的全新部門 我們正站在 科技的最前緣 致力於塑造數位行銷領域的未來 我們的主要...,The Pocket CompanyThe Pocket Company Accucrazy...
4,Data Engineer - Python,Data Engineer,作為數據工程師（Python），您將負責在我們的G123平台上設計和實施強大的數據流程，以確...,作為數據工程師 您將負責在我們的 平台上設計和實施強大的數據流程 以確保最佳的時效性 數據品...,Python G G G
5,NLP Research Scientist,NLP,作為自然語言處理（NLP）研究技術人員，您將負責在與工程團隊密切合作的同時實施自然語言處理技...,作為自然語言處理 研究技術人員 您將負責在與工程團隊密切合作的同時實施自然語言處理技術 隨著...,NLP G NLP
6,MLOps Engineer,MachineLearning,作為機器學習運維（MLOps）工程師，您將負責實現模型在生產系統中的自動化部署，以在G123...,作為機器學習運維 工程師 您將負責實現模型在生產系統中的自動化部署 以在 平台不斷擴展其遊戲...,MLOps G CI CD
7,Data Scientist,data s,作為數據科學家，您將負責基於我們的G123平台大數據，應用機器學習、統計分析和數據可視化技術...,作為數據科學家 您將負責基於我們的 平台大數據 應用機器學習 統計分析和數據可視化技術 結合...,G G
8,Prompt Engineer,Prompt,- 與人工智慧和數據科學團隊緊密合作，為AI語言模型（如GPT-4）或生成式AI模型（如St...,與人工智慧和數據科學團隊緊密合作 為 語言模型 如 或生成式 模型 如 和 創建和優化提示 ...,AI GPT AI Stable Diffusion Midjourney AI AI
9,CV Research Scientist,ComputerVision,作為計算機視覺研究技術人員，您將負責在我們的內部橫幅生成工具中實現諸如GAN或穩定擴散等新的...,作為計算機視覺研究技術人員 您將負責在我們的內部橫幅生成工具中實現諸如 或穩定擴散等新的計算...,GAN G


In [6]:
# 分五大類
data_dict = {'NLP工程師':'自然語言|NLP|nlp|LLM|語言模型',
             '資料工程師':'ETL|Data Engineer|數據工程|Data Pipeline|倉儲|Hadoop',
             '資料科學家':'Data Scientist|深度學習|影像辨識|電腦視覺|人工智慧|AI|Deep Learning|影像處理|時間序列|Time Series|數據建模|資料科學|機器學習|ComputerVision|Computer Vision|Image Processing|MLOps|Machine Learning',
             '演算法工程師':'演算法|Algorithm|算法開發|算法設計',
             '數據分析師':'資料分析|商業數據分析|BI|Business Intelligence|商業分析|數據|Data Analyst'}

#############   
# def categorize_data(df, data_dict, label_column_name, tag_column_name):
#     # 創建一個新的標籤欄位
#     df[tag_column_name] = ""

#     # 對每個類別和相關的標籤執行分類
#     for category, keywords in data_dict.items():
#         for keyword in keywords.split('|'):
#             # 使用正則表達式檢查是否有包含關鍵字
#             df.loc[df[label_column_name].str.contains(keyword, case=False, regex=True), tag_column_name] = category

#     return df


# df = pd.DataFrame(data)
# result_df = categorize_data(PB_data_df_test, data_dict, '個人整體標籤', 'tag')
# result_df
############# 

def categorize_data(df, category_dict, label_column_names, tag_column_name):
    # 創建一個新的標籤欄位
    df[tag_column_name] = ""

    # 對每一行執行分類
    def categorize_row(row):
        for label_column_name in label_column_names:
            for category, keywords in category_dict.items():
                for keyword in keywords.split('|'):
                    # 使用正則表達式檢查是否有包含關鍵字
                    if pd.notna(row[label_column_name]) and re.search(keyword, row[label_column_name], re.IGNORECASE):
                        return category
        return ""

    # 將分類標籤應用於每一行
    df[tag_column_name] = df.apply(categorize_row, axis=1)

    return df

# classify each job with duty
label_column_names = ['標籤', '職稱', '職缺描述']
result_df = categorize_data(new_demand, data_dict, label_column_names, '職務')
result_df


,職稱,標籤,職缺描述,職缺描述（中文）,職缺描述（英文）,職務
0,資料科學家,"AI, Data Scientist, Statistics Analysis",中醫只有華人才能精確掌握，雖然目前還很低調，但我們能提供的產品價值是過往廠商都無法提供的。除...,中醫只有華人才能精確掌握 雖然目前還很低調 但我們能提供的產品價值是過往廠商都無法提供的 除...,Insight automated data pipelines,資料科學家
1,[總部] 機器學習主任工程師 Principal Engineer_0220*,"軟體開發, AI, 軟體設計工程師, 機器學習, 大數據, 演算法開發工程師",我們是集團的策略長室團隊，目前正在積極尋找Machine Learning 主任工程師 加入...,我們是集團的策略長室團隊 目前正在積極尋找 主任工程師 加入我們 你將負責 應用架構設計與實...,Machine Learning Gamania Data Center is an exp...,資料科學家
2,Backend Engineer (Python),-,【Please Apply Here】 are on a mission to spark ...,,Please Apply Here are on a mission to spark co...,資料工程師
3,AI 產品經理( Prompt Engineering/ Fine Tuning）,-,部門介紹The Pocket CompanyThe Pocket Company是Accuc...,部門介紹 是 的全新部門 我們正站在 科技的最前緣 致力於塑造數位行銷領域的未來 我們的主要...,The Pocket CompanyThe Pocket Company Accucrazy...,資料科學家
4,Data Engineer - Python,Data Engineer,作為數據工程師（Python），您將負責在我們的G123平台上設計和實施強大的數據流程，以確...,作為數據工程師 您將負責在我們的 平台上設計和實施強大的數據流程 以確保最佳的時效性 數據品...,Python G G G,資料工程師
5,NLP Research Scientist,NLP,作為自然語言處理（NLP）研究技術人員，您將負責在與工程團隊密切合作的同時實施自然語言處理技...,作為自然語言處理 研究技術人員 您將負責在與工程團隊密切合作的同時實施自然語言處理技術 隨著...,NLP G NLP,NLP工程師
6,MLOps Engineer,MachineLearning,作為機器學習運維（MLOps）工程師，您將負責實現模型在生產系統中的自動化部署，以在G123...,作為機器學習運維 工程師 您將負責實現模型在生產系統中的自動化部署 以在 平台不斷擴展其遊戲...,MLOps G CI CD,資料科學家
7,Data Scientist,data s,作為數據科學家，您將負責基於我們的G123平台大數據，應用機器學習、統計分析和數據可視化技術...,作為數據科學家 您將負責基於我們的 平台大數據 應用機器學習 統計分析和數據可視化技術 結合...,G G,資料科學家
8,Prompt Engineer,Prompt,- 與人工智慧和數據科學團隊緊密合作，為AI語言模型（如GPT-4）或生成式AI模型（如St...,與人工智慧和數據科學團隊緊密合作 為 語言模型 如 或生成式 模型 如 和 創建和優化提示 ...,AI GPT AI Stable Diffusion Midjourney AI AI,NLP工程師
9,CV Research Scientist,ComputerVision,作為計算機視覺研究技術人員，您將負責在我們的內部橫幅生成工具中實現諸如GAN或穩定擴散等新的...,作為計算機視覺研究技術人員 您將負責在我們的內部橫幅生成工具中實現諸如 或穩定擴散等新的計算...,GAN G,資料科學家


In [7]:
result_df = result_df[result_df['職務']!='']
result_df

,職稱,標籤,職缺描述,職缺描述（中文）,職缺描述（英文）,職務
0,資料科學家,"AI, Data Scientist, Statistics Analysis",中醫只有華人才能精確掌握，雖然目前還很低調，但我們能提供的產品價值是過往廠商都無法提供的。除...,中醫只有華人才能精確掌握 雖然目前還很低調 但我們能提供的產品價值是過往廠商都無法提供的 除...,Insight automated data pipelines,資料科學家
1,[總部] 機器學習主任工程師 Principal Engineer_0220*,"軟體開發, AI, 軟體設計工程師, 機器學習, 大數據, 演算法開發工程師",我們是集團的策略長室團隊，目前正在積極尋找Machine Learning 主任工程師 加入...,我們是集團的策略長室團隊 目前正在積極尋找 主任工程師 加入我們 你將負責 應用架構設計與實...,Machine Learning Gamania Data Center is an exp...,資料科學家
2,Backend Engineer (Python),-,【Please Apply Here】 are on a mission to spark ...,,Please Apply Here are on a mission to spark co...,資料工程師
3,AI 產品經理( Prompt Engineering/ Fine Tuning）,-,部門介紹The Pocket CompanyThe Pocket Company是Accuc...,部門介紹 是 的全新部門 我們正站在 科技的最前緣 致力於塑造數位行銷領域的未來 我們的主要...,The Pocket CompanyThe Pocket Company Accucrazy...,資料科學家
4,Data Engineer - Python,Data Engineer,作為數據工程師（Python），您將負責在我們的G123平台上設計和實施強大的數據流程，以確...,作為數據工程師 您將負責在我們的 平台上設計和實施強大的數據流程 以確保最佳的時效性 數據品...,Python G G G,資料工程師
5,NLP Research Scientist,NLP,作為自然語言處理（NLP）研究技術人員，您將負責在與工程團隊密切合作的同時實施自然語言處理技...,作為自然語言處理 研究技術人員 您將負責在與工程團隊密切合作的同時實施自然語言處理技術 隨著...,NLP G NLP,NLP工程師
6,MLOps Engineer,MachineLearning,作為機器學習運維（MLOps）工程師，您將負責實現模型在生產系統中的自動化部署，以在G123...,作為機器學習運維 工程師 您將負責實現模型在生產系統中的自動化部署 以在 平台不斷擴展其遊戲...,MLOps G CI CD,資料科學家
7,Data Scientist,data s,作為數據科學家，您將負責基於我們的G123平台大數據，應用機器學習、統計分析和數據可視化技術...,作為數據科學家 您將負責基於我們的 平台大數據 應用機器學習 統計分析和數據可視化技術 結合...,G G,資料科學家
8,Prompt Engineer,Prompt,- 與人工智慧和數據科學團隊緊密合作，為AI語言模型（如GPT-4）或生成式AI模型（如St...,與人工智慧和數據科學團隊緊密合作 為 語言模型 如 或生成式 模型 如 和 創建和優化提示 ...,AI GPT AI Stable Diffusion Midjourney AI AI,NLP工程師
9,CV Research Scientist,ComputerVision,作為計算機視覺研究技術人員，您將負責在我們的內部橫幅生成工具中實現諸如GAN或穩定擴散等新的...,作為計算機視覺研究技術人員 您將負責在我們的內部橫幅生成工具中實現諸如 或穩定擴散等新的計算...,GAN G,資料科學家


In [8]:
category_counts = result_df['職務'].value_counts()
category_counts

資料科學家     31
資料工程師     15
數據分析師      7
NLP工程師     3
演算法工程師     3
Name: 職務, dtype: int64

In [9]:
result_df.to_excel('Yourator數據職缺JD_中英文分離_1004.xlsx', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
cd '/content/gdrive/My Drive/data'

In [ ]:
test.to_excel('Yourator.xlsx',index=False)